### This strategy is based on the paper "Facial Emotion Recognition Based on Biorthogonal Wavelet Entropy, Fuzzy Support Vector Machine, and Stratified Cross Validation"

In [ ]:
import os               # VSCode Navigator
import subprocess       # Terminal Access
import pandas as pd

video_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/VideoFlash'
mp4_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/mp4'
output_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/output'


os.makedirs(mp4_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


# A list of all the files
flv_video_files = [video for video in os.listdir(video_dir) if video.endswith('.flv')][:2500]

for video in flv_video_files:

    # This converts the .flv file into mp4 files for OpenFace
    mp4_video_name = video.replace(".flv", ".mp4")
    subprocess.run([
        'ffmpeg',
        '-i',
        os.path.join(video_dir, video),
        '-y',
        '-loglevel',
        'error',
        os.path.join(mp4_dir, mp4_video_name)])
    

    # subprocess.run([
    # "docker", "run", "--rm",
    # "--platform", "linux/amd64",  # explicitly force amd64 emulation
    # "-v", "/Users/songye/Desktop/Dev/REU/CREMA-D:/data",
    # "algebr/openface:latest",
    # "/home/openface-build/build/bin/FeatureExtraction",
    # "-f", "/data/mp4/" + mp4_video_name,
    # "-out_dir", "/data/output/"])

    # This did not work, so have to run Docker manually via Terminal
    # docker run -it -v /Users/songye/Desktop/Dev/REU/CREMA-D:/data algebr/openface:latest
    # for f in /data/mp4/*.mp4; do /home/openface-build/build/bin/FeatureExtraction -f "$f" -out_dir /data/output/; done

    

# Construct Feature Matrix

In [ ]:
# Now we need a feature matrix
# Each row = one entire video, columns = aggregated AU statistics + emotion label (Each AU's mean and std)
list_of_csvs = [csv for csv in os.listdir(output_dir) if csv.endswith(".csv")]
feature_matrix = pd.DataFrame()

count = 0
cols_order = []
#emotion_count = {}

for csv in list_of_csvs:
    df = pd.read_csv(os.path.join(output_dir, csv))
    df.columns = df.columns.str.strip()

    list_means = []
    list_std = []
    list_max = []

    for col in df.columns:
        if col.startswith('AU') and (col.endswith('_r') or col.endswith('_c')):
            list_means.append(df[col].mean())
            list_std.append(df[col].std())
            list_max.append(df[col].max())
            if count == 0:
                cols_order.append(col)

    total_list = list_means.copy()
    total_list.extend(list_std)
    total_list.extend(list_max)

    # To get the emotion from the file name
    emotion = csv.split('_')[2]
    total_list.append(emotion)

    feature_matrix = pd.concat([feature_matrix, pd.DataFrame([total_list])], ignore_index=True)
    count += 1

col_names = ([col + '_mean' for col in cols_order] + 
             [col + '_std' for col in cols_order] + 
             [col + '_max' for col in cols_order] +
             ['Emotion'])

feature_matrix.columns = col_names
#feature_matrix.info

# Imports

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

print(feature_matrix["Emotion"].value_counts())

Emotion
DIS    322
ANG    307
SAD    304
FEA    298
HAP    269
NEU    236
Name: count, dtype: int64


# Happy Model

In [29]:
feature_matrix["target_score"] = -1
for index,row in feature_matrix.iterrows():
    if row["Emotion"] == "HAP":
        feature_matrix.loc[index, "target_score"] = 1


X = feature_matrix.drop(columns = ["Emotion", "target_score"])
y = feature_matrix["target_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
happy_scaler = StandardScaler()
X_train_scaled = happy_scaler.fit_transform(X_train)
X_test_scaled = happy_scaler.transform(X_test)

happy_model = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced', probability=True)

happy_model.fit(X_train_scaled, y_train)
y_pred = happy_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.9655172413793104
              precision    recall  f1-score   support

          -1       0.98      0.98      0.98       294
           1       0.89      0.89      0.89        54

    accuracy                           0.97       348
   macro avg       0.93      0.93      0.93       348
weighted avg       0.97      0.97      0.97       348



# Neutral Model

In [30]:
feature_matrix["target_score"] = -1
for index,row in feature_matrix.iterrows():
    if row["Emotion"] == "NEU":
        feature_matrix.loc[index, "target_score"] = 1


X = feature_matrix.drop(columns = ["Emotion", "target_score"])
y = feature_matrix["target_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
neutral_scaler = StandardScaler()
X_train_scaled = neutral_scaler.fit_transform(X_train)
X_test_scaled = neutral_scaler.transform(X_test)



neutral_model = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced', probability=True)

neutral_model.fit(X_train_scaled, y_train)
y_pred = neutral_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.8103448275862069
              precision    recall  f1-score   support

          -1       0.96      0.82      0.88       301
           1       0.40      0.77      0.52        47

    accuracy                           0.81       348
   macro avg       0.68      0.79      0.70       348
weighted avg       0.88      0.81      0.83       348



# Disgust Model

In [31]:
feature_matrix["target_score"] = -1
for index,row in feature_matrix.iterrows():
    if row["Emotion"] == "DIS":
        feature_matrix.loc[index, "target_score"] = 1


X = feature_matrix.drop(columns = ["Emotion", "target_score"])
y = feature_matrix["target_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
disgust_scaler = StandardScaler()
X_train_scaled = disgust_scaler.fit_transform(X_train)
X_test_scaled = disgust_scaler.transform(X_test)

disgust_model = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced', probability=True)

disgust_model.fit(X_train_scaled, y_train)
y_pred = disgust_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.882183908045977
              precision    recall  f1-score   support

          -1       0.97      0.88      0.92       283
           1       0.63      0.88      0.74        65

    accuracy                           0.88       348
   macro avg       0.80      0.88      0.83       348
weighted avg       0.91      0.88      0.89       348



# Anger Model

In [32]:
feature_matrix["target_score"] = -1
for index,row in feature_matrix.iterrows():
    if row["Emotion"] == "ANG":
        feature_matrix.loc[index, "target_score"] = 1


X = feature_matrix.drop(columns = ["Emotion", "target_score"])
y = feature_matrix["target_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
anger_scaler = StandardScaler()
X_train_scaled = anger_scaler.fit_transform(X_train)
X_test_scaled = anger_scaler.transform(X_test)

anger_model = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced', probability=True)

anger_model.fit(X_train_scaled, y_train)
y_pred = anger_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.8132183908045977
              precision    recall  f1-score   support

          -1       0.90      0.87      0.88       286
           1       0.48      0.56      0.52        62

    accuracy                           0.81       348
   macro avg       0.69      0.72      0.70       348
weighted avg       0.83      0.81      0.82       348



# Sadness Model

In [33]:
feature_matrix["target_score"] = -1
for index,row in feature_matrix.iterrows():
    if row["Emotion"] == "SAD":
        feature_matrix.loc[index, "target_score"] = 1


X = feature_matrix.drop(columns = ["Emotion", "target_score"])
y = feature_matrix["target_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sadness_scaler = StandardScaler()
X_train_scaled = sadness_scaler.fit_transform(X_train)
X_test_scaled = sadness_scaler.transform(X_test)

sadness_model = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced', probability=True)

sadness_model.fit(X_train_scaled, y_train)
y_pred = sadness_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.7126436781609196
              precision    recall  f1-score   support

          -1       0.91      0.72      0.81       287
           1       0.34      0.67      0.45        61

    accuracy                           0.71       348
   macro avg       0.63      0.70      0.63       348
weighted avg       0.81      0.71      0.74       348



# Fear Model

In [34]:
feature_matrix["target_score"] = -1
for index,row in feature_matrix.iterrows():
    if row["Emotion"] == "FEA":
        feature_matrix.loc[index, "target_score"] = 1


X = feature_matrix.drop(columns = ["Emotion", "target_score"])
y = feature_matrix["target_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
fear_scaler = StandardScaler()
X_train_scaled = fear_scaler.fit_transform(X_train)
X_test_scaled = fear_scaler.transform(X_test)


fear_model = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale', class_weight = 'balanced', probability=True)

fear_model.fit(X_train_scaled, y_train)
y_pred = fear_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.7758620689655172
              precision    recall  f1-score   support

          -1       0.94      0.78      0.85       288
           1       0.42      0.75      0.54        60

    accuracy                           0.78       348
   macro avg       0.68      0.77      0.69       348
weighted avg       0.85      0.78      0.80       348



# Prediction using all models

In [44]:
trained_models = {
    "HAP" : happy_model,
    "DIS" : disgust_model,
    "SAD" : sadness_model,
    "FEA" : fear_model,
    "NEU" : neutral_model,
    "ANG" : anger_model
}

trained_scaler = {
    "HAP" : happy_scaler,
    "DIS" : disgust_scaler,
    "SAD" : sadness_scaler,
    "FEA" : fear_scaler,
    "NEU" : neutral_scaler,
    "ANG" : anger_scaler
}

emotions = ["HAP", "DIS", "SAD", "FEA", "NEU", "ANG"]

def predict_emotion(feature_vector):
    max_confidence = 0
    predicted_emotion = emotions[0]

    for emotion in emotions:
        scaled_vector = trained_scaler[emotion].transform([feature_vector])
        if trained_models[emotion].predict_proba(scaled_vector)[0][1] > max_confidence:
            max_confidence = trained_models[emotion].predict_proba(scaled_vector)[0][1]
            predicted_emotion = emotion
        
    return predicted_emotion


# Applying to CREMA-D Feature Matrix

In [45]:
correct_count = 0

for index, row in feature_matrix.iterrows():
    feature_vector = row.drop(["Emotion", "target_score"]).values
    predicted_emotion = predict_emotion(feature_vector)
    if row["Emotion"] == predicted_emotion:
        correct_count += 1

accuracy = correct_count / len(feature_matrix)
print(accuracy)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/

0.8179723502304147


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/